# Where the model is confidently wrong

Browses the **validation** images ranked by how confidently the model is wrong
(from `runs/ensemble_dinov3_raw_full_448/posthoc/val_worst_wrong.csv`, produced by
`UncertaintyChallenge2026/analyze_val_failures.py`).

- **confidence** = max softmax probability (how sure the model was)
- **y** = true class (remapped contiguous index), **pred** = predicted class
- **original** = the class id in the original WILDS space (`new_to_original`)
- **p_true** = probability the model gave the *actual* class (0 = complete miss)

The widgets let you pick a ranking, filter by in/out-of-distribution domain, and
step through the images one at a time.

## 1. Configuration

Edit the paths if your layout differs.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from PIL import Image as PILImage

RUN_DIR = Path("/home/alice/work/dtu_ss_26/runs/ensemble_dinov3_raw_full_448")
DATA_ROOT = Path("/home/alice/work/dtu_ss_26/challenge_data")

CSV_PATH = RUN_DIR / "posthoc" / "val_worst_wrong.csv"
IMG_DIR = DATA_ROOT / "val" / "images"

class_map = json.loads((DATA_ROOT / "class_mapping.json").read_text())
new_to_orig = {int(k): int(v) for k, v in class_map["new_to_original"].items()}

## 2. Load the ranked wrong predictions

If the CSV is missing, run `analyze_val_failures.py` first (it reuses the cached
val logits/embeddings, so it is fast and needs no GPU).

In [ ]:
df = pd.read_csv(CSV_PATH)
df["original_y"] = df["y"].map(new_to_orig)
df["original_pred"] = df["pred"].map(new_to_orig)
df = df[["uid", "domain", "y", "original_y", "pred", "original_pred",
         "confidence", "p_true", "nll", "brier", "cluster"]]
print(f"{len(df)} wrong predictions on val")
df.head()

## 3. Interactive viewer

Controls:

- **Sort by** — `confidence` (confidently wrong) or `p_true` (model gave almost
  no probability to the true class, i.e. biggest miss).
- **Domain** — `all`, or only `id` / `ood` images.
- **Index** — which image in the ranked list to show (0 = most wrong).
- The image is displayed with its rank, uid, domain, class indices, and the
  model's confidence.

In [ ]:
sort_by = widgets.Dropdown(
    options=[("confidence (confidently wrong)", "confidence"),
             ("p_true (biggest miss)", "p_true")],
    value="confidence", description="Sort by:",
)
domain = widgets.Dropdown(
    options=[("all", "all"), ("id", "id"), ("ood", "ood")],
    value="all", description="Domain:",
)
rank = widgets.IntSlider(min=0, max=max(len(df) - 1, 0), step=1, value=0,
                         description="Index:", layout=widgets.Layout(width="600px"))
out = widgets.Output()


def _filtered():
    d = df if domain.value == "all" else df[df["domain"] == domain.value]
    return d.sort_values(sort_by.value, ascending=(sort_by.value == "p_true"))


def update_rank(_=None):
    rank.max = max(len(_filtered()) - 1, 0)
    if rank.value > rank.max:
        rank.value = rank.max


sort_by.observe(update_rank, "value")
domain.observe(update_rank, "value")


def show(_=None):
    d = _filtered().reset_index(drop=True)
    row = d.iloc[rank.value]
    img = PILImage.open(IMG_DIR / f"{row['uid']}.jpg").convert("RGB")
    with out:
        out.clear_output(wait=True)
        display(img)
        print(f"rank #{rank.value} of {len(d)} | {row['uid']}")
        print(f"domain={row['domain']}   cluster={row['cluster']}")
        print(f"true y={row['y']} (orig {row['original_y']})   "
              f"pred={row['pred']} (orig {row['original_pred']})")
        print(f"confidence={row['confidence']:.4f}   p_true={row['p_true']:.4f}   "
              f"nll={row['nll']:.2f}   brier={row['brier']:.2f}")


rank.observe(show, "value")
show()

controls = widgets.VBox([sort_by, domain, rank])
display(widgets.HBox([controls, out]))